# MATRICE DE CONFUSION

---
## Imports

In [1]:
# Gestion DataFrame & Dataviz
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
# Imports des modèles de scikit-learn
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
# Import pour la division train/test 
from sklearn.model_selection import train_test_split
# Import pour les métriques de matrice de confusion
from sklearn import metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:
# Import du dataset
titanic = sns.load_dataset('titanic')
# Nombres de lignes et de colonnes dans le DataFrame
print(f"{titanic.shape[0]} lignes et {titanic.shape[1]} colonnes")
# Affichage des données
titanic.head(3)

891 lignes et 15 colonnes


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True


---
## Exploratory Data Analyst 

### **`Objectif :`** 
- Tester des modèles de classification sur un jeu de données simple
- Extraire et comparer les scores des différents modèles
- Visualiser la comparaison en dataviz

Le but ici n'est pas d'entraîner des modèles afin d'avoir le meilleur score.  
On cherche à obtenir un visuel de matrice corrélation multi-modèle

In [3]:
# On ne garde que les Series pclass, sex, age en se séparant des valeurs nulles.
titanic = titanic[['pclass', 'sex', 'age','survived']]
# On observe le jeu de données.
# Des valeurs nulles sont présentes dans la colonne `age`.
# la colonne sex présente des valeurs catégorielle (str).
titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    891 non-null    int64  
 1   sex       891 non-null    str    
 2   age       714 non-null    float64
 3   survived  891 non-null    int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 32.1 KB


In [4]:
# Ssuppression des lignes qui présentes des valeurs nulles.
titanic = titanic.dropna()
# Définition des features (X) et de la target (y).
X = titanic[['pclass', 'sex', 'age',]]
y = titanic['survived']
# Encodage de la Serie `sex` pour obtenir des valeurs numérique
X['sex'] = X.sex.factorize()[0]
# Vérification de la taille et dimensions des features et target
print(X.shape, y.shape)

In [ ]:
# Définition des modèles utilisés.
svc = SVC()
sgd = SGDClassifier()
knn = KNeighborsClassifier()
lr = LogisticRegression()
# Liste des modèles défini
model_list = [svc, sgd, lr, knn]

In [ ]:
# Division du dataset - entrainement / test
X_train, X_test, y_train, y_test = train_test_split(
            X, y,             # Dicision sur les features et la target
            test_size=0.2,    # Division 80/20 - Paréto
            random_state=42,  # Paramètre de reproductibilité défini à 42
            stratify=y)       # stratification de la target pour garder le même équilibre dans les jeu de données entrainement / test

# Dataviz des résultats
# Définition d'une visualisation multiple
fig, ax = plt.subplots(1, 4, figsize=(20,3))
# Boucle sur la liste des modèles en récupérant l'index (enumerate())
for index, model in enumerate(model_list):
    # Entrainement des modèles
    model.fit(X_train, y_train)
    # Définition du titre -> modèle + score 
    ax[index].set_title(f"{model}: {model.score(X_test, y_test):.1%}")
    # Définition de la prediction dans une variable
    y_pred = model.predict(X_test)
    # Configuration de la matrice de confusion
    metrics.ConfusionMatrixDisplay(
            confusion_matrix=metrics.confusion_matrix(y_test, y_pred),
            # Configuration du visuel
            display_labels=['Mort', 'Survivant']).plot(ax=ax[index])